# **OPTIMIZING EV CHARGING PLACEMENT (OBJ 3)**

# 1. Import libraries and data

In [253]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from scipy.spatial import cKDTree


In [254]:
df = pd.read_csv("final_data.csv")
df.head()


,ID,Station Address,Total Number of Chargers,DC,AC,Total Station Capacity (KW),Total Charger Bays,DC Charger Bays,AC Charger Bays,DC_Capacity_kW,...,Latitude,Longitude,Avg_AC_Hours,Avg_DC_Hours,AC_revenue (RM),DC_revenue (RM),Total_revenue (RM),population,ADT,Peak_Hour
0,3,"Mukim Hulu Lepar, Daerah Kuantan, 26300 Negeri...",1,1,0,50,1,1,0,50,...,3.809554,103.160203,0.00,2.65,0.00,132.50,132.50,49.4,26873,No
1,6,"Blh Ecoenergy Sdn. Bhd., A1, Tingkat Bawah, Lo...",2,2,0,44,2,2,0,44,...,3.841799,103.336868,0.00,2.77,0.00,121.88,121.88,61.2,37159,No
2,25,Petronas R&R Gambang Lebuhraya Pantai Timur (E...,2,2,0,180,2,2,0,180,...,3.718008,103.124958,0.00,3.07,0.00,552.60,552.60,29.0,26454,No
3,57,"Ground Floor, Berjaya Megamall, Lot G15, Jalan...",2,0,2,33,2,0,2,0,...,3.815411,103.330127,9.63,0.00,174.78,0.00,174.78,29.0,16420,No
4,58,"Pl- Hadapan Bangunan Mbk Jalan Tanah Putih, Ta...",2,0,2,33,2,0,2,0,...,3.797762,103.322072,9.05,0.00,164.26,0.00,164.26,29.0,28990,No


# 2. Feature engineering

In [255]:
df = df.dropna(subset=["Latitude", "Longitude"])

In [256]:
#build spatial index
coords = df[["Latitude", "Longitude"]].values
tree = cKDTree(coords)

In [257]:
#find the distance to the nearest station
distances, _ = tree.query(coords, k=3)
df["dist_nearest_station_km"] = distances[:, 1] * 111


In [258]:
#station density within radius
#calculate the density of EV charging stations by determining how many other stations are located within a 3 kilometer radius of each station
def count_within_radius(point, radius_km=3):
    radius_deg = radius_km / 111
    return len(tree.query_ball_point(point, radius_deg)) - 1

df["stations_within_3km"] = df[["Latitude","Longitude"]].apply(
    lambda x: count_within_radius(x.values, 3), axis=1
)


In [259]:
#Local Demand Intensity
#measure the local demand intensity at each location
#The formula used is: (population * ADT) / (stations_within_3km + 1)

df["local_demand_index"] = (
    df["population"] * df["ADT"]
) / (df["stations_within_3km"] + 1)


In [260]:
df[["Station Address","dist_nearest_station_km", "stations_within_3km", "local_demand_index"]].head()
#higher local_demand_index means the specific area have higher demand for new ev stations

,Station Address,dist_nearest_station_km,stations_within_3km,local_demand_index
0,"Mukim Hulu Lepar, Daerah Kuantan, 26300 Negeri...",10.888673,0,1.327526e+06
1,"Blh Ecoenergy Sdn. Bhd., A1, Tingkat Bawah, Lo...",0.696738,10,2.067392e+05
2,Petronas R&R Gambang Lebuhraya Pantai Timur (E...,10.888673,0,7.671660e+05
3,"Ground Floor, Berjaya Megamall, Lot G15, Jalan...",0.220775,17,2.645444e+04
4,"Pl- Hadapan Bangunan Mbk Jalan Tanah Putih, Ta...",0.079407,16,4.945353e+04


In [261]:
#Feature selection
features = [
    "population",
    "ADT",
    "Total Charger Bays",
    "AC",
    "DC",
    "dist_nearest_station_km",
    "stations_within_3km"
    #'local_demand_index' is the target variable, so it's removed from features
]

X = df[features]
y = df["local_demand_index"]

# 3. Data partitioning

In [263]:
from sklearn.model_selection import LeaveOneOut, cross_val_score

loo = LeaveOneOut()
# Use cross_val_score to get a more reliable average error
scores = cross_val_score(model, X, y, cv=loo, scoring='neg_mean_absolute_error')
print("Average MAE:", -scores.mean())

Average MAE: 593129.3638225067


# 4. Feature scaling

In [264]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. Training model

## Support Vector Regression




In [265]:
svr = SVR(kernel='rbf', C=10, epsilon=0.1)
loo = LeaveOneOut()

In [266]:
# This builds 34 different models and records the 'out-of-sample' prediction for each
y_pred_loo = cross_val_predict(model, X_scaled, y, cv=loo)

In [267]:
# Calculate Metrics based on LOOCV
mae = mean_absolute_error(y, y_pred_loo)
rmse = np.sqrt(mean_squared_error(y, y_pred_loo))
r2 = r2_score(y, y_pred_loo)

print("--- LOOCV Evaluation Metrics ---")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

--- LOOCV Evaluation Metrics ---
MAE:  593129.4134
RMSE: 1366245.2683
R²:   -0.1340


### Log transformation

In [268]:
# Transform the target to minimize error
y_log = np.log1p(df["local_demand_index"])

# Run LOOCV on the LOG target
y_pred_log = cross_val_predict(model, X_scaled, y_log, cv=loo)

In [269]:
# Calculate Metrics based on LOOCV
mae = mean_absolute_error(y, y_pred_loo)
rmse = np.sqrt(mean_squared_error(y, y_pred_loo))
# Calculate R2 on the log scale
r2_log = r2_score(y_log, y_pred_log)

print("--- LOOCV Evaluation Metrics ---")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Log-Scale R²: {r2_log:.4f}")

--- LOOCV Evaluation Metrics ---
MAE:  593129.4134
RMSE: 1366245.2683
Log-Scale R²: 0.8783


### Model fitting

In [270]:
# Final model fit
svr.fit(X_scaled, y)

SVR(C=10)

## Random Forest

In [271]:
# max_depth=3: Keeps trees simple to prevent overfitting on 34 rows
# n_estimators=100: Number of trees to average
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,
    random_state=42
)

In [272]:
# Perform LOOCV predictions
loo = LeaveOneOut()
y_pred_rf = cross_val_predict(rf_model, X, y, cv=loo)

In [273]:
mae_rf = mean_absolute_error(y, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y, y_pred_rf))
r2_rf = r2_score(y, y_pred_rf)

print("--- Random Forest LOOCV Metrics ---")
print(f"MAE:  {mae_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"R²:   {r2_rf:.4f}")

--- Random Forest LOOCV Metrics ---
MAE:  443870.9166
RMSE: 1170863.9648
R²:   0.1671


### Hyperparameter tuning

In [274]:
from sklearn.model_selection import GridSearchCV, LeaveOneOut, cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# 1. Setup LOOCV and Parameter Grid
loo = LeaveOneOut()

# We test small depths to prevent overfitting on your 34 rows
param_grid = {
    'n_estimators': [100],
    'max_depth': [2, 3, 4],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', None]
}

# 2. Run Grid Search
# This will search 36 combinations, each cross-validated 34 times
rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=loo,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X, y)

# 3. Extract Best Model
best_rf = grid_search.best_estimator_

# 4. Final Re-Evaluation (LOOCV with best parameters)
y_pred_tuned = cross_val_predict(best_rf, X, y, cv=loo)

# 5. Calculate Final Metrics
tuned_r2 = r2_score(y, y_pred_tuned)
tuned_mae = mean_absolute_error(y, y_pred_tuned)

print("--- Optimized Random Forest Results ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Improved R²:     {tuned_r2:.4f}")
print(f"Improved MAE:    {tuned_mae:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


--- Optimized Random Forest Results ---
Best Parameters: {'max_depth': 2, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Improved R²:     0.2808
Improved MAE:    506455.3106


### Model fitting

In [275]:
# Fit final model
best_rf.fit(X, y)

RandomForestRegressor(max_depth=2, max_features='sqrt', random_state=42)

# 6. Generate candidate locations using K-Means

In [278]:
# 1. Define the boundaries of Kuantan
lat_min, lat_max = df['Latitude'].min(), df['Latitude'].max()
lon_min, lon_max = df['Longitude'].min(), df['Longitude'].max()

# 2. Create a grid of 100 potential "New" points
lat_grid = np.linspace(lat_min, lat_max, 10)
lon_grid = np.linspace(lon_min, lon_max, 10)
grid_points = [(lat, lon) for lat in lat_grid for lon in lon_grid]

# 3. Convert to DataFrame
candidates = pd.DataFrame(grid_points, columns=['Latitude', 'Longitude'])

# 4. Filter: Remove points too close to existing stations (Avoid oversupply)
# We want 'Optimal' to mean 'Filling a gap'
candidate_coords = candidates[["Latitude", "Longitude"]].values
dist, _ = tree.query(candidate_coords, k=1)
candidates["dist_nearest_station_km"] = dist * 111

# Only keep points that are at least 2km away from an existing station
candidates = candidates[candidates["dist_nearest_station_km"] > 2.0].reset_index(drop=True)

# 5. Sample down to your 15 desired locations
candidates = candidates.sample(15, random_state=42).reset_index(drop=True)

In [279]:
# Find the nearest known data point for each new candidate to estimate Population and ADT
# This ensures your SVR has the 'Kuantan context' for these new spots
_, indices = tree.query(candidates[["Latitude", "Longitude"]].values, k=1)

candidates["population"] = df.iloc[indices]["population"].values
candidates["ADT"] = df.iloc[indices]["ADT"].values
candidates["stations_within_3km"] = [count_within_radius(x, 3) for x in candidates[["Latitude", "Longitude"]].values]

# Re-apply your spec allocation
candidates[['Total Charger Bays', 'AC', 'DC']] = candidates.apply(
    lambda x: pd.Series(allocate_specs(x)), axis=1
)

# 7. Predicting revenue using SVR model

In [280]:
# Prediction using SVR
X_candidates = candidates[features]
X_candidates_scaled = scaler.transform(X_candidates)

# Use your SVR model (ensure it's the one you fitted on the full dataset)
candidates["predicted_revenue"] = svr.predict(X_candidates_scaled)

In [281]:
best_locations = candidates.sort_values(
    "predicted_revenue", ascending=False
).reset_index(drop=True)

In [282]:
candidates["predicted_revenue_rm"] = candidates["predicted_revenue"].apply(lambda x: f"RM {x:,.2f}")

In [283]:
candidates.drop(columns=['predicted_revenue'], inplace=True)
candidates

,Latitude,Longitude,dist_nearest_station_km,population,ADT,stations_within_3km,Total Charger Bays,AC,DC,predicted_revenue_rm
0,3.916329,103.347163,2.094435,19.2,12775,0,2,1,1,"RM 208,710.20"
1,3.817168,103.251932,5.615575,133.0,69067,-1,6,2,4,"RM 208,725.03"
2,4.015489,103.220189,17.128859,133.0,52021,-1,6,2,4,"RM 208,724.77"
3,4.164229,103.315419,9.878180,19.2,14513,-1,2,1,1,"RM 208,743.18"
4,3.718008,103.156702,3.523527,29.0,26454,-1,2,1,1,"RM 208,723.02"
5,3.866749,103.156702,6.360475,49.4,26873,-1,2,1,1,"RM 208,740.01"
6,3.916329,103.315419,5.566940,19.2,12775,-1,2,1,1,"RM 208,733.56"
7,4.065069,103.251932,18.002387,19.2,14513,-1,2,1,1,"RM 208,729.29"
8,3.767588,103.156702,4.674388,49.4,26873,-1,2,1,1,"RM 208,732.47"
9,3.965909,103.188445,14.223348,133.0,52021,-1,6,2,4,"RM 208,724.78"


# 8. Predicting total cars (based on ADT)

In [284]:
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# 1. Define Features (X) and the New Target (y = ADT)
# We exclude ADT from the features because it's now what we want to predict!
traffic_features = ["Latitude", "Longitude", "population", "dist_nearest_station_km"]
X_traffic = df[traffic_features]
y_traffic = df["ADT"]

In [286]:
# 2. Scaling is MANDATORY for SVR
scaler_X_traffic = StandardScaler()
scaler_y_traffic = StandardScaler()

In [287]:
X_traffic_scaled = scaler_X_traffic.fit_transform(X_traffic)
# Reshape y because the scaler expects a 2D array
y_traffic_scaled = scaler_y_traffic.fit_transform(y_traffic.values.reshape(-1, 1)).flatten()

In [288]:
# 3. Train the SVR Model for Traffic Prediction
# We use 'rbf' kernel to capture non-linear traffic patterns
traffic_svr = SVR(kernel='rbf', C=100, epsilon=0.1)
traffic_svr.fit(X_traffic_scaled, y_traffic_scaled)

SVR(C=100)

In [289]:
# Prepare Candidate Features for the Traffic Model
X_cand_traffic = candidates[traffic_features]
X_cand_traffic_scaled = scaler_X_traffic.transform(X_cand_traffic)

In [290]:
# Predict and Inverse Scale
# The prediction comes out in scaled units, so we turn it back into "Number of Cars"
predicted_traffic_scaled = traffic_svr.predict(X_cand_traffic_scaled)
candidates["predicted_cars"] = scaler_y_traffic.inverse_transform(predicted_traffic_scaled.reshape(-1, 1))

In [291]:
# 6. Final Formatting
# Round to the nearest whole car
candidates["predicted_cars"] = candidates["predicted_cars"].round(0).astype(int)

print("--- Predicted Traffic (ADT) for Candidate Locations ---")
print(candidates[["Latitude", "Longitude", "predicted_cars"]].head())

--- Predicted Traffic (ADT) for Candidate Locations ---
   Latitude   Longitude  predicted_cars
0  3.916329  103.347163            9146
1  3.817168  103.251932           51695
2  4.015489  103.220189           43760
3  4.164229  103.315419           40492
4  3.718008  103.156702           49017


# 9. Calculating utilisation rate

In [292]:
# --- Updated 2026 Context ---
ev_penetration = 0.0254  # 2.54% based on Q1 2026 National Data by JPJ
capture_rate = 0.03      # Conservative 3% (Standard for regional hubs like Kuantan)
avg_charge_time = 45     # Minutes per session
total_bays = 6

# Daily EV Arrivals
# Of the predicted cars, how many are EVs AND stop to charge?
candidates["daily_ev_arrivals"] = (
    candidates["predicted_cars"] * ev_penetration * capture_rate
)

# Total Minutes of Charging Demand
demand_mins = candidates["daily_ev_arrivals"] * avg_charge_time

# Total Minutes of Supply (6 bays available for 24 hours)
supply_mins = total_bays * 24 * 60

# Final Utilisation Rate
candidates["utilisation_rate"] = (demand_mins / supply_mins) * 100
candidates["utilisation_rate"] = candidates["utilisation_rate"].round(2)

In [294]:
candidates["daily_ev_arrivals"] = candidates["daily_ev_arrivals"].round(2)

# 10. Calculate final score to rank the candidates

In [295]:
import numpy as np

# Perform normalization to get the standardized value

# Convert 'predicted_revenue_rm' back to numeric for calculations
candidates['predicted_revenue_numeric'] = candidates['predicted_revenue_rm'].astype(str).str.replace('RM ', '').str.replace(',', '').astype(float)

# Normalize both scores to a 0-1 scale so they can be compared fairly
candidates["norm_revenue"] = (candidates["predicted_revenue_numeric"] - candidates["predicted_revenue_numeric"].min()) / (candidates["predicted_revenue_numeric"].max() - candidates["predicted_revenue_numeric"].min())
candidates["norm_util"] = (candidates["utilisation_rate"] - candidates["utilisation_rate"].min()) / (candidates["utilisation_rate"].max() - candidates["utilisation_rate"].min())

# Calculate a Weighted Score (e.g., 70% Revenue importance, 30% Utilisation)
candidates["final_score"] = (candidates["norm_revenue"] * 0.7) + (candidates["norm_util"] * 0.3)

# Rank based on the Final Score
best_locations = candidates.sort_values("final_score", ascending=False)

In [299]:
best_locations = best_locations[best_locations['dist_nearest_station_km'] > 3]
final_candidates = best_locations[['Latitude', 'Longitude', 'predicted_revenue_rm', 'predicted_cars', 'daily_ev_arrivals', 'utilisation_rate', 'final_score']]
final_candidates

,Latitude,Longitude,predicted_revenue_rm,predicted_cars,daily_ev_arrivals,utilisation_rate,final_score
3,4.164229,103.315419,"RM 208,743.18",40492,30.85,16.07,0.897571
14,3.767588,103.220189,"RM 208,744.32",36853,28.08,14.63,0.895382
5,3.866749,103.156702,"RM 208,740.01",40263,30.68,15.98,0.830937
12,4.065069,103.378906,"RM 208,741.68",27200,20.73,10.79,0.773014
8,3.767588,103.156702,"RM 208,732.47",45772,34.88,18.17,0.715147
7,4.065069,103.251932,"RM 208,729.29",43873,33.43,17.41,0.636407
1,3.817168,103.251932,"RM 208,725.03",51695,39.39,20.52,0.604250
10,4.164229,103.220189,"RM 208,726.88",43932,33.48,17.44,0.587497
6,3.916329,103.315419,"RM 208,733.56",18987,14.47,7.54,0.548699
13,4.065069,103.124958,"RM 208,724.77",43935,33.48,17.44,0.544209


# 11. Geographical map

In [302]:
import folium
import pandas as pd

# Calculate the mean latitude and longitude to center the map
center_lat = best_locations['Latitude'].mean()
center_lon = best_locations['Longitude'].mean()

# Create a Folium map centered around the candidate locations
candidate_map = folium.Map(location=[center_lat, center_lon], zoom_start=12)

# Function to determine marker color and status based on final_score quantiles
def get_location_status(final_score, df):
    # Define thresholds based on the distribution of final_scores
    # Top ~33% are green (good), next ~33% are blue (medium), bottom ~33% are red (not recommended)
    score_threshold_green = df['final_score'].quantile(0.66) # Locations with score >= this are in top ~33%
    score_threshold_blue = df['final_score'].quantile(0.33)  # Locations with score >= this (and < green threshold) are in middle ~33%

    if final_score >= score_threshold_green:
        return 'green', 'High priority (Tier 1)'
    elif final_score >= score_threshold_blue:
        return 'blue', 'Strategic priority (Tier 2)'
    else:
        return 'red', 'Low priority (Tier 3)'

# Add markers for each candidate location
# Function to generate a clean, horizontal popup
def create_popup_html(row, rank):
    # Determine Status based on your indicator logic
    color, status = get_location_status(row['final_score'], best_locations)

    html = f"""
    <div style="font-family: 'Arial', sans-serif; min-width: 220px; padding: 5px;">
        <h4 style="margin: 0 0 10px 0; color: #333; border-bottom: 2px solid {color}; padding-bottom: 5px;">
            Rank #{rank}: <span style="color: {color};">{status}</span>
        </h4>
        <table style="width: 100%; border-collapse: collapse; font-size: 13px;">
            <tr>
                <td style="padding: 4px 0; color: #666; font-weight: bold;">Final Score:</td>
                <td style="padding: 4px 0; text-align: right; font-weight: bold;">{row['final_score']:.4f}</td>
            </tr>
            <tr style="border-top: 1px solid #eee;">
                <td style="padding: 4px 0; color: #666;">Predicted Rev:</td>
                <td style="padding: 4px 0; text-align: right; font-weight: bold;">{row['predicted_revenue_rm']}</td>
            </tr>
            <tr style="border-top: 1px solid #eee;">
                <td style="padding: 4px 0; color: #666;">Utilisation:</td>
                <td style="padding: 4px 0; text-align: right; color: #2c3e50; font-weight: bold;">{row['utilisation_rate']}%</td>
            </tr>
            <tr style="border-top: 1px solid #eee;">
                <td style="padding: 4px 0; color: #666;">Daily Traffic:</td>
                <td style="padding: 4px 0; text-align: right; font-weight: bold;">{int(row['predicted_cars']):,} cars</td>
            </tr>
            <tr style="border-top: 1px solid #eee;">
                <td style="padding: 4px 0; color: #666;">Charger Setup:</td>
                <td style="padding: 4px 0; text-align: right; font-weight: bold;">{int(row['AC'])} AC | {int(row['DC'])} DC</td>
            </tr>
        </table>
    </div>
    """
    return html

# Update your marker loop
for index, row in best_locations.iterrows():
    if pd.notnull(row['Latitude']) and pd.notnull(row['Longitude']):
        color, _ = get_location_status(row['final_score'], best_locations)

        # Call the new HTML function
        popup_content = create_popup_html(row, index + 1)

        folium.Marker(
            location=[row['Latitude'], row['Longitude']],
            popup=folium.Popup(popup_content, max_width=300),
            icon=folium.Icon(color=color, icon='bolt', prefix='fa')
        ).add_to(candidate_map)

# Display the map
candidate_map

Indicators:

Green - High priority (Tier 1)

Blue - Strategic priority (Tier 2)

Red - Low priority (Tier 3)